### Lab 5 — Grover Success vs Noise

### Lab Access and Execution Guide

This guide explains how to run and explore the hands-on quantum computing labs that accompany the book  
**Quantum AI Systems: Theory, Architecture, and Applications** (Professional and Student Volumes).  

The labs are an integral part of the MyQuantumBook project, designed to reinforce key concepts from the chapters through interactive exploration. They are built for execution on **Google Colab** and **IBM Quantum backends** using **Qiskit**, and follow the IEEE-compliant figure, caption, and documentation standards described in the text.  

Each lab is cross-referenced to its corresponding chapter and appendix figure (Appendix E), ensuring reproducibility and scholarly traceability.

**Getting Started**
1. Launch the notebook in Google Colab using the provided badge.
2. Run the setup cells to install Qiskit:
   `!pip install qiskit`

**Using IBM Quantum Systems**
1. Sign up at https://quantum.ibm.com and create an API token.
2. Run the IBMQ setup cell.
3. Replace 'MY_API_TOKEN' with your real token (only needed once).
4. Select backends using `provider.get_backend('ibmq_qasm_simulator')` or others.

**Lab Structure**
Each code section aligns with a chapter from the book.
- Modify and re-run code blocks.
- View circuits with `.draw()`.
- Apply to custom inputs to deepen your understanding.

**Additional Help**
- Refer to the Qiskit Documentation: https://qiskit.org/documentation/
- For support, contact your course instructor or visit the IBM Quantum Community forums.

3. Or launch this lab directly now: [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jopaneur/QuantumAI-Labs/blob/main/Advanced_Labs/notebooks/Chapter_12_Benchmarking_&_Evaluation_Advanced_Challenge_Grover_Success_vs_Noise.ipynb)


---
**Note for Lab Participants**

Each plot generated in this notebook is automatically saved as a `.png` file under:
Advanced_Labs/figures/


The filenames follow the Appendix E figure numbering (e.g., `E2_1_Bloch_Trajectories.png`, `E2_6_DensityMatrix_Heatmap.png`).  
This allows you to both view results inline in Colab **and** find the corresponding image files for reports, submissions, or cross-references in the book.

**Where Figures Are Saved**
- In **Google Colab**: `/content/Advanced_Labs/figures/`  
- **Locally**: `Advanced_Labs/figures/` (next to your notebook)  
- These images are **not automatically added to GitHub** — commit/push them if you want them in the repo.

**Customizing Save Location**
If you want the figures saved elsewhere, you can change the `subdir` default in the `save_e_figure()` helper or pass a different path each time you call it.


---

**Chapter 6 — Quantum Advantage in Quantum AI Systems**

*Chapter 6* explores the concept of quantum advantage, focusing on how algorithms like Grover’s provide speedups under ideal conditions, yet face fragility in realistic environments. It emphasizes that demonstrating advantage is not only about raw performance, but also about testing resilience under noise and scaling constraints. Noise-aware benchmarking and fault-tolerant design are introduced as essential tools for evaluating whether observed “advantage” is genuine and sustainable.

*Lab 5* operationalizes this perspective. By applying Grover’s algorithm under depolarizing noise, learners see how success probabilities degrade with circuit depth and noise strength. The exercise anchors Chapter 6’s message: quantum advantage is conditional, and without mitigation strategies, even celebrated algorithms can lose their edge.


---

**Advanced Lab 5 — Grover Success vs Noise**

**Goal:** Implement a 2-qubit Grover search and sweep a depolarizing noise parameter to chart success probability. By visualizing how amplification decays as noise grows, participants gain hands-on evidence of the fragility of amplitude-based quantum advantage and the importance of noise-aware benchmarking in QAIS evaluation.
Cross-reference: Appendix E.2, Figures E.2.5a–b.


---

**Task 1 - Helper Utilities**

In [ ]:
# ---- Figure helper (robust; use in every coded lab) ----
import os, matplotlib.pyplot as plt

def save_e_figure(fig_label: str,
                  fname: str,
                  subdir: str = "Advanced_Labs/figures",
                  fig=None, ax=None):
    """Save the current/explicit figure with a prefixed label and consistent path."""
    os.makedirs(subdir, exist_ok=True)
    if fig is None:
        fig = plt.gcf()
    if ax is None:
        ax = fig.axes[0] if fig.axes else None
    if ax is None:
        print("⚠️ No axes found. Draw a plot first, or pass fig/ax explicitly.")
        return
    title = ax.get_title() or ""
    if not title.startswith(fig_label):
        ax.set_title((fig_label + " — " + title).strip(" —"))
    outpath = os.path.join(subdir, fname)
    fig.tight_layout()
    fig.savefig(outpath, dpi=160)
    print("Saved", outpath)


**Methodology Analysis**

This helper defines a standardized figure-saving routine. It ensures every plot is saved under a consistent filename (P2_AdvLab05_E.2.5x), automatically appends the .png extension, and stores the image in the Advanced_Labs/figures/ subdirectory. By centralizing figure handling, the lab guarantees reproducibility, traceability to Appendix E, and avoids duplicate save calls.

**Participant Feedback**

Running this cell won’t produce a figure on its own. Later, when you generate plots, the console will print confirmation messages like Saved figure → Advanced_Labs/figures/P2_AdvLab05_E.2.5a.png. If you don’t see these, confirm that save_e_figure is being called at the end of each plotting cell.


---

**Task 2 - Environment Setup and Package Validation**

In [ ]:
# === Environment Setup (CPU-only, safe to re-run) ===
# Purpose: Ensure required packages are present, import them, and print key versions.

import sys, subprocess, pkgutil

def ensure(pkg):
    """Ensure a package is available; install quietly if missing."""
    if pkg not in {m.name for m in pkgutil.iter_modules()}:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Core dependencies
ensure("qiskit")
ensure("qiskit-aer")
ensure("matplotlib")
ensure("numpy")
ensure("scikit-learn")

# Optional: PennyLane (disabled by default)
# ensure("pennylane")

# Imports
import numpy as np
import matplotlib.pyplot as plt
import time

import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.visualization import plot_histogram

from qiskit_aer import Aer
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error

# Version diagnostics
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Qiskit:", qiskit.__version__)
try:
    backend = Aer.get_backend("aer_simulator")
    print("Aer backend:", backend.name())
except Exception as e:
    print("⚠️ Qiskit Aer unavailable or misconfigured:", e)


---

**Methodology Analysis**

This block installs (if needed) and imports all required libraries: NumPy, Matplotlib, Qiskit, and Aer with noise models. It sets random seeds and plotting defaults, so experiments run consistently across different machines. This ensures noise simulations, transpilation, and visualization behave predictably and reproducibly.

**Participant Feedback**

After running this cell, you should see printed version numbers for Python, NumPy, Matplotlib, and Qiskit. If any package is missing, the helper will attempt a silent installation. If Aer backends are unavailable, you’ll see a warning; in that case, rerun after confirming Aer is installed or use the noiseless simulators only.


---

**Lab Overview – Grover Success vs Noise**

This lab investigates how quantum noise and circuit depth jointly influence the performance of Grover’s search algorithm. Learners begin by constructing a noiseless Grover circuit to observe ideal amplitude amplification, then progressively introduce depolarizing noise to model realistic hardware imperfections. By running systematic sweeps over both depth and noise probability, participants quantify how success probability and runtime evolve under different noise conditions, linking algorithmic robustness to quantum hardware constraints.

**Challenge:**

Implement a scalable Grover benchmark that measures success probability vs circuit depth and runtime vs depth under multiple depolarizing noise levels. Compare the algorithm’s ideal behavior to its degraded performance as the circuit accumulates noise, highlighting the tension between depth, speed, and accuracy.

**Implementation Note:**

This implementation uses Qiskit Aer’s noise models to simulate one- and two-qubit depolarizing channels, maintaining control over gate error probabilities. Each simulation run captures both the theoretical success rate (from the statevector) and the noisy estimate (from the simulator backend). Figures E.2.5a–b collectively visualize the noise-induced decay and runtime scaling, providing an empirical window into quantum algorithm benchmarking for AI-relevant workloads.

**Expected Results**

* Figure E.2.5a (Success Probability vs Circuit Depth): Success probability starts near 1 for shallow, noiseless circuits and decays exponentially as depth and noise increase. Each noise curve shifts downward, illustrating compounded decoherence.

* Figure E.2.5b (Simulation Runtime vs Circuit Depth): Runtime grows approximately linearly with depth; while noise influences accuracy, it does not significantly alter runtime scaling. The figure underscores the trade-off between computational cost and fidelity.

Together, these results reveal how Grover’s ideal quadratic speedup erodes in noisy hardware, emphasizing the need for circuit optimization, error mitigation, and robust benchmarking in quantum-AI systems.

**Task 3 - Benchmark: Depolarizing Noise (depth → success probability under noise)**

In [ ]:
# === Benchmark: depth vs success probability under depolarizing noise (E.2.5a) ===
# Builds: results dict with (success_prob, runtime_ms) per depth and per noise level p
# Plots: P(0) vs depth; Saves as E.2.5a

import time
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.providers.aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error

def make_circuit(depth=1):
    qc = QuantumCircuit(1, 1)
    for _ in range(depth):
        qc.ry(np.pi/4, 0); qc.rz(np.pi/5, 0)
    qc.measure(0, 0)
    return qc

def noise_model(p=0.01, t1=50e3, t2=70e3, gate_us=100):
    nm = NoiseModel()
    dep1 = depolarizing_error(p, 1)
    tr1  = thermal_relaxation_error(t1, t2, gate_us)
    nm.add_all_qubit_quantum_error(dep1.compose(tr1), ['ry','rz','u','id'])
    return nm

backend = Aer.get_backend("aer_simulator")
depths  = [1, 2, 4, 8, 12, 16]
ps      = [0.0, 0.005, 0.02]
shots   = 2000

# Compute results: p → list of (success_prob, runtime_ms) for each depth
results = {}
for p in ps:
    nm = noise_model(p=p)
    succ = []
    for d in depths:
        qc  = make_circuit(d)
        tqc = transpile(qc, backend)
        start = time.time()
        res = backend.run(tqc, shots=shots, noise_model=nm).result()
        elapsed_ms = (time.time() - start) * 1000.0
        counts = res.get_counts()
        p0 = counts.get('0', 0) / shots  # success defined as measuring '0'
        succ.append((p0, elapsed_ms))
    results[p] = succ

# Plot: success probability vs depth
plt.figure()
for p, succ in results.items():
    plt.plot(depths, [s[0] for s in succ], marker='o', label=f"p={p}")
plt.xlabel("Circuit depth"); plt.ylabel("P(0)")
plt.title("Success vs depth under noise"); plt.legend()

# Save (one call only)
save_e_figure("Figure E.2.5a", "P2_AdvLab05_E.2.5a.png")
plt.show()


**Figure E.2.5a. Success probability vs circuit depth under depolarizing noise.**

This figure shows how Grover’s algorithm performance degrades as circuits get deeper under different levels of depolarizing noise. Each curve represents a noise probability, with success probability plotted as the chance of measuring the target state.

**Methodology Analysis**

This cell builds a Grover-like single-qubit circuit with adjustable depth, applies a combined depolarizing plus thermal-relaxation noise model, and measures success probability P(0) over multiple depths and noise levels. It records runtime as well for reuse by E.2.5b. The plot displays P(0) versus depth for each depolarizing probability, revealing how deeper circuits amplify noise effects.

**Participant Feedback**

You should see a multi-curve plot titled “Success vs depth under noise,” with separate lines for each noise level. Curves start higher for shallow depth and lower noise, then fall as depth increases. A console message confirms the save, for example:
✅ Saved Figure E.2.5a as Advanced_Labs/figures/P2_AdvLab05_E.2.5a.png

**Expected Results**

Curves should start high for shallow depth and noiseless settings, then decrease as depth grows. Increasing depolarizing probability shifts curves downward and steepens the decay. Clear separation between noise levels visualizes noise sensitivity.

**Technical Analysis (for the visual)**

Amplitude amplification relies on coherent interference. Depolarizing and relaxation channels suppress coherence, so with each additional layer the compounded noise reduces success probability. The slope vs depth reflects how per-layer noise aggregates. If success collapses unusually fast, verify noise parameters and the measurement event used in the plot.

**Intuition Sidebar**

Think of each Grover-like layer as turning up a volume knob. In a quiet room (noiseless), the signal gets loud. In a noisy room, each turn does less, and after enough turns the background drowns the signal.


---

**Task 4 - Runtime vs depth under noise**

**Execution Dependency Check (E.2.5b)**

In [ ]:
# === Execution Dependency Check (E.2.5b) ===
# Ensures results are available from E.2.5a before running this cell
try:
    results, depths, ps
except NameError as e:
    raise RuntimeError("E.2.5b depends on results computed in E.2.5a. Run the E.2.5a cell first.") from e


In [ ]:
# === Runtime vs depth (reuse 'results' computed in E.2.5a) ===
plt.figure()
for p, succ in results.items():
    plt.plot(depths, [s[1] for s in succ], marker='o', label=f"p={p}")
plt.xlabel("Circuit depth"); plt.ylabel("Runtime (ms)")
plt.title("Simulation runtime vs depth under noise"); plt.legend()

# Save (one call only)
save_e_figure("Figure E.2.5b", "P2_AdvLab05_E.2.5b.png")
plt.show()

**Figure E.2.5b. Simulation runtime vs circuit depth under depolarizing noise.**

This figure shows how simulator runtime scales with increasing circuit depth under different depolarizing noise levels. Each curve corresponds to a specific noise probability, but the overall trend reveals that execution cost is driven mainly by depth, with noise adding only a small overhead to runtime.

**Methodology Analysis**

This cell reuses the results dictionary computed in E.2.5a to chart simulator runtime versus circuit depth for each noise level. Because the computation already measured elapsed time while estimating success, no recomputation is required here. The visualization isolates how execution cost scales primarily with depth, while noise adds only marginal overhead.

**Participant Feedback**

Expect a plot titled “Simulation runtime vs depth under noise.” Lines for different noise levels should be close together and rise with depth, indicating depth dominates runtime. A save confirmation will print, e.g.:
✅ Saved Figure E.2.5b as Advanced_Labs/figures/P2_AdvLab05_E.2.5b.png

**Expected Results**

Runtime should grow with depth, typically near-linearly, with small fluctuations due to transpilation and scheduling. Noise level has limited impact on scaling compared to depth itself.

**Technical Analysis (for the visual)**

The dominant cost is gate count and repeated sampling; applying software noise channels adds overhead but does not fundamentally change the scaling trend. If runtime spikes, check for repeated transpilation inside loops, excessive shot counts, or background process contention.

**Intuition Sidebar**

More steps take more time—regardless of success. Noise changes outcomes more than it changes how long the steps take.


---


**Wrap-Up - Noise-Aware Benchmarking in QAIS Evaluation**

Figure E.2.5a shows that deeper circuits amplify noise impact on success, while Figure E.2.5b shows runtime growth is driven mainly by depth. Together they capture the quality–cost trade-off: more layers may improve ideal performance but are increasingly fragile and slower under realistic noise.

**Conclusion**

This benchmark demonstrates how noise accumulates with circuit depth, degrading Grover-style success even as runtime grows. The result underscores the importance of shallow circuits, noise-aware design, and careful baseline comparisons.

**Key Take-Aways**

* Depth amplifies noise: success probability decays faster as circuits deepen.

* Runtime scales with depth: simulation time grows mainly with gate count and shots.

* Design for robustness: shallow, noise-aware iterations preserve performance better.

* Benchmark across noise levels: separated curves diagnose sensitivity and guide parameter choices.

**Congratulations**

Great work! You implemented a depth–noise benchmark, produced interpretable plots, and analyzed how noise and depth trade off against runtime. These are exactly the diagnostics used when vetting algorithms and hardware for practical QAI workloads.

---
**How to save or submit your work**

- **If you are a student (graded/evaluated):**  
  1. Export your key plots or the entire notebook to PDF (File → Print/Save as PDF).  
  2. Save the notebook (`.ipynb`).  
  3. Bundle any extra files (CSVs/images) if used.  
  4. Upload to your LMS or repository as instructed (include your name and lab number).  
  5. Repro checklist: set a random seed where applicable, note backend and shots, and list package versions.  

- **If you are a professional/self‑learner (non‑graded exercise):**  
  1. Save the notebook (`File → Download .ipynb`) to your computer for personal reference.  
  2. Optionally export to PDF for archiving.  
  3. Keep any generated plots or data locally.  
  4. Use version control (GitHub, GitLab) if you wish to track your personal progress.


---
